In [77]:
# Import necessary libraries and suppress warnings
import warnings
warnings.filterwarnings('ignore')

import pandas                  as pd
import numpy                   as np
import matplotlib.pyplot       as plt
import seaborn                 as sns
import statsmodels.api         as sm
import statsmodels.formula.api as smf

# This statement allow to display plot without asking to 
%matplotlib inline

# always make it pretty 
plt.style.use('ggplot')

In [78]:
df = pd.read_csv("Data_MLB_2025_StatcastPostseason_PitchByPitch_20251102a.csv")

print(f'Rows: {df.shape[0]}, Columns: {df.shape[1]}')
print(list(df.columns))

Rows: 14096, Columns: 95
['game_year', 'game_date', 'player_name', 'pitcher', 'batter', 'balls', 'strikes', 'inning', 'inning_topbot', 'at_bat_number', 'pitch_number', 'events', 'pitch_type', 'pitch_name', 'release_speed', 'release_pos_x', 'release_pos_z', 'description', 'spin_dir', 'spin_rate_deprecated', 'break_angle_deprecated', 'break_length_deprecated', 'zone', 'des', 'game_type', 'stand', 'p_throws', 'home_team', 'away_team', 'type', 'hit_location', 'bb_type', 'pfx_x', 'pfx_z', 'plate_x', 'plate_z', 'on_3b', 'on_2b', 'on_1b', 'outs_when_up', 'hc_x', 'hc_y', 'tfs_deprecated', 'tfs_zulu_deprecated', 'fielder_2', 'fielder_2.1', 'umpire', 'sv_id', 'vx0', 'vy0', 'vz0', 'ax', 'ay', 'az', 'sz_top', 'sz_bot', 'hit_distance_sc', 'launch_speed', 'launch_angle', 'effective_speed', 'release_spin_rate', 'release_extension', 'game_pk', 'fielder_2.2', 'fielder_2.3', 'fielder_3', 'fielder_4', 'fielder_5', 'fielder_6', 'fielder_7', 'fielder_8', 'fielder_9', 'release_pos_y', 'estimated_ba_using_sp

The following are the columns of the dataset.

**game_year**: Year it took place

**game_date**: Date of the game

**pitch_type**: The type of pitch derived from Statcast

**events**: Event of the resulting plate appearance

**description**: Description of the resulting pitch

**launch_speed**: Exit velocity of the batted ball as tracked by Statcast. For batted balls not tracked directly, estimates are included based on Statcast's estimation process

**pitch_name**: The name of the pitch derived from the Statcast data

**home_score**: Pre-pitch home team score

**away_score**: Pre-pitch away team score

**bat_score**: Pre-pitch batting team score

**fld_score**: Pre-pitch fielding team score

**if_fielding_alignment**: Infield fielding alignment at the time of the pitch

**of_fielding_alignment**: Outfield fielding alignment at the time of the pitch

**spin_axis**: Spin axis in the 2D X–Z plane in degrees from 0–360 (180 represents pure backspin, 0 represents pure topspin)

**effective_speed**: Derived speed based on the extension of the pitcher's release

**release_spin_rate**: Release spin rate (TBD in original source)

**release_extension**: Release extension of pitch in feet as tracked by Statcast

**release_pos_y**: Release position of pitch measured in feet from the catcher's perspective

**at_bat_number**: Plate appearance number within the game

**player_name**: Player's name tied to the event

**batter**: MLB player ID for the batter tied to the play event

**pitcher**: MLB player ID for the pitcher tied to the play event

**pfx_x**: Horizontal movement in feet from the catcher's perspective

**release_speed**: Pitch velocity. Velocities from 2008–16 are via Pitch F/X (adjusted); velocities from 2017 onward are Statcast out-of-hand measurements

**release_pos_x**: Horizontal release position in feet from the catcher's perspective

**release_pos_z**: Vertical release position in feet from the catcher's perspective

**spin_dir**: Deprecated field from the older tracking system

**zone**: Zone location of the ball when it crosses the plate from the catcher's perspective

**p_throws**: Hand the pitcher throws with

**stand**: Batter's side of the plate (left/right)

**balls**: Pre-pitch number of balls in the count

**strikes**: Pre-pitch number of strikes in the count

**pfx_z**: Vertical movement in feet from the catcher's perspective

**plate_x**: Horizontal position of the ball as it crosses home plate from the catcher's perspective

**plate_z**: Vertical position of the ball as it crosses home plate from the catcher's perspective

**vx0**: Velocity of the pitch in x-dimension (ft/s), determined at y=50 ft

**vy0**: Velocity of the pitch in y-dimension (ft/s), determined at y=50 ft

**vz0**: Velocity of the pitch in z-dimension (ft/s), determined at y=50 ft

**ax**: Acceleration in x-dimension (ft/s^2), determined at y=50 ft

**ay**: Acceleration in y-dimension (ft/s^2), determined at y=50 ft

**az**: Acceleration in z-dimension (ft/s^2), determined at y=50 ft

## Data Types & Duplicates

In [79]:
print("Data types:")
print(df.dtypes.value_counts().to_string())
print(f"Exact duplicate rows: {df.duplicated().sum()}")
print(f"game_date dtype: {df['game_date'].dtype} - needs parsing to datetime")

Data types:
float64    48
int64      30
object     17
Exact duplicate rows: 0
game_date dtype: object - needs parsing to datetime


In [80]:
cols_dropped = list(df.columns[df.isna().all()])
print(f'Columns to drop: \n{list(df.columns[df.isna().all()])}')

df.drop(columns=cols_dropped, inplace=True)


Columns to drop: 
['spin_dir', 'spin_rate_deprecated', 'break_angle_deprecated', 'break_length_deprecated', 'tfs_deprecated', 'tfs_zulu_deprecated', 'umpire', 'sv_id']


## Filter to Blue Jays Pitching Only

When Toronto is the **home** team, their pitchers throw in the **Top** of the inning. When Toronto is the **away** team, their pitchers throw in the **Bot** of the inning.

In [81]:
bj_df = df[
    ((df["home_team"] == "TOR") & (df["inning_topbot"] == "Top")) |
    ((df["away_team"] == "TOR") & (df["inning_topbot"] == "Bot"))
]

print(f"Blue Jays pitching rows: {bj_df.shape[0]:}")
print(f"Unique games: {bj_df['game_pk'].nunique()}")
print(f"Unique pitchers (by ID): {bj_df['pitcher'].nunique()}")


Blue Jays pitching rows: 2788
Unique games: 18
Unique pitchers (by ID): 15


## Missing Values — Blue Jays Pitching Subset

In [82]:
null = bj_df.isnull().sum()
null_percent = (bj_df.isnull().sum() / len(bj_df) * 100).round(2)
null_df = pd.DataFrame({"missing_count": null, "missing_percent": null_percent})
null_df = null_df[null_df["missing_count"] > 0].sort_values("missing_percent", ascending=False)
print(f"Columns with missing values: {len(null_df)} out of {bj_df.shape[1]}\n")
null_df

Columns with missing values: 20 out of 87



,missing_count,missing_percent
on_3b,2599,93.22
estimated_ba_using_speedangle,2340,83.93
launch_speed_angle,2340,83.93
bb_type,2337,83.82
hc_x,2337,83.82
hc_y,2337,83.82
on_2b,2315,83.03
hit_location,2188,78.48
estimated_woba_using_speedangle,2092,75.04
woba_denom,2090,74.96
